In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import re
import spacy
import json
import unicodedata
from gensim.models import Phrases
from gensim.models.phrases import Phraser


# =====================================================
# Configuration
# =====================================================
DATA_PATH = (
    r"C:\Users\55279\Desktop\Mestrado\0.Thesis\Codes\Common\NTSB_ALL.csv"
)
GLOSSARY_PATH = (
    r"C:\Users\55279\Desktop\Mestrado\0.Thesis\Codes\Common\Glossary.json"
)
# =====================================================
# NLTK setup
# =====================================================

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")

STOPWORDS_LIST = set(stopwords.words("english"))

# =====================================================
# Load data
# =====================================================

NTSB_df = pd.read_csv(DATA_PATH)

# =====================================================
# Abbreviation expansion
# =====================================================

with open(GLOSSARY_PATH, "r", encoding="utf-8") as f:
    ABBREVIATIONS = json.load(f)


def expand_abbreviations(text, abbr_dict):
    if pd.isna(text):
        return text

    text = str(text)

    for abbr, full in abbr_dict.items():
        pattern = r"\b" + re.escape(abbr) + r"\b"
        text = re.sub(pattern, full, text, flags=re.IGNORECASE)

    return text


NTSB_df["narrative"] = (
    NTSB_df["narrative"]
    .astype(str)
    .apply(lambda x: expand_abbreviations(x, ABBREVIATIONS))
)

# =====================================================
# NLP preprocessing
# =====================================================
nlp_full = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def preprocess_text(text):

    if pd.isna(text):
        return []

    text = str(text)

    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^A-Za-z\s]", " ", text)

    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode()

    text = text.lower().strip()

    doc = nlp_full(text)

    tokens = [
        tok.lemma_
        for tok in doc
        if tok.lemma_
        and tok.lemma_ not in STOPWORDS_LIST
        and tok.is_alpha
        and len(tok.lemma_) >= 2
    ]

    return tokens


NTSB_df["tokens"] = NTSB_df["narrative"].apply(preprocess_text)
NTSB_df = NTSB_df[NTSB_df["tokens"].astype(bool)]


# =====================================================
# N-gram modeling
# =====================================================

bigram = Phrases(NTSB_df["tokens"], min_count=100, threshold=15)
trigram = Phrases(bigram[NTSB_df["tokens"]], min_count=75, threshold=20)

bigram_mod = Phraser(bigram)
trigram_mod = Phraser(trigram)

NTSB_df["tokens_bigrams"] = NTSB_df["tokens"].apply(lambda x: bigram_mod[x])
NTSB_df["tokens_trigrams"] = (
    NTSB_df["tokens_bigrams"].apply(lambda x: trigram_mod[x])
)


STOPWORDS_AFTER = {
    "around", "back", "by", "first", "no", "not", "off", "on",
    "out", "over", "take", "top", "up", "aircraft", "airplane",
    "acft", "flight", "flt", "time", "fly"
}

NTSB_df["tokens_final"] = NTSB_df["tokens_trigrams"].apply(
    lambda toks: [t for t in toks if t not in STOPWORDS_AFTER]
)


# Create bigrams and trigrams
bigram = Phrases(NTSB_df['tokens'], min_count=60, threshold=15)
trigram = Phrases(bigram[NTSB_df['tokens']], min_count=55, threshold=20)

bigram_mod = Phraser(bigram)
trigram_mod = Phraser(trigram)
#fourgram_mod = Phraser(fourgram)

NTSB_df['tokens_bigrams'] = NTSB_df['tokens'].apply(lambda x: bigram_mod[x])
NTSB_df['tokens_trigrams'] = NTSB_df['tokens_bigrams'].apply(lambda x: trigram_mod[x])

# Define stopwords to remove AFTER bigram/trigram creation
stopwords_after_phrases = {
    "around", "back", "by", "first", "no", "not", "off", "on",
    "out", "over", "take", "top", "up", "aircraft", "airplane",
    "acft", "flight", "flt", "time", "fly"
}
NTSB_df["tokens_final"] = NTSB_df["tokens_trigrams"].apply(
    lambda toks: [t for t in toks if t not in STOPWORDS_AFTER]
)
